# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a Croissant-described dataset using the `mlcroissant` library.

We will work from the Croissant schema URL directly, inspect the dataset metadata, discover available record sets and fields using their `@id` values, and then load selected record sets into pandas DataFrames for analysis.

### Dataset Source
Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Important note on identifiers
In this notebook, dataset entities are referenced by their Croissant `@id` wherever possible. This includes record sets and fields discovered from the metadata.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset from the Croissant URL and inspect the top-level metadata.

The `dataset.metadata` object should be treated as a metadata object. To work with a serializable representation, we convert it with `.to_json()`.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Convert metadata to a JSON-serializable structure
metadata_json = dataset.metadata.to_json()

print("Dataset loaded successfully.\n")
print("Name:", metadata_json.get("name", "N/A"))
print("Version:", metadata_json.get("version", "N/A"))
print("Identifier:", metadata_json.get("identifier", "N/A"))
print("License:", metadata_json.get("license", "N/A"))
print("Date published:", metadata_json.get("datePublished", "N/A"))
print("\nDescription:\n", metadata_json.get("description", "N/A"))

In [ ]:
# A compact metadata summary
summary = {
    "name": metadata_json.get("name"),
    "identifier": metadata_json.get("identifier"),
    "version": metadata_json.get("version"),
    "license": metadata_json.get("license"),
    "keywords": metadata_json.get("keywords", []),
}

summary

## 2. Data Overview
In this section, we inspect the dataset structure and list available record sets and fields.

Because Croissant datasets can contain multiple record sets, the first step is to discover what is available programmatically. We will build helper functions that search through the JSON representation of the metadata and collect entities with `@id` values.

This is especially useful for large or nested metadata documents.

In [ ]:
from collections.abc import Mapping, Sequence

def collect_entities(obj, entity_type=None, results=None):
    """Recursively collect JSON-LD entities, optionally filtering by @type."""
    if results is None:
        results = []

    if isinstance(obj, Mapping):
        obj_type = obj.get("@type")
        if "@id" in obj:
            if entity_type is None:
                results.append(obj)
            else:
                if obj_type == entity_type or (isinstance(obj_type, list) and entity_type in obj_type):
                    results.append(obj)
        for value in obj.values():
            collect_entities(value, entity_type=entity_type, results=results)
    elif isinstance(obj, Sequence) and not isinstance(obj, (str, bytes)):
        for item in obj:
            collect_entities(item, entity_type=entity_type, results=results)
    return results

all_entities = collect_entities(metadata_json)
all_entities_by_id = {}
for entity in all_entities:
    entity_id = entity.get("@id")
    if entity_id and entity_id not in all_entities_by_id:
        all_entities_by_id[entity_id] = entity

print(f"Discovered {len(all_entities_by_id)} unique JSON-LD entities with @id values.")

In [ ]:
# Try to discover record sets and fields from metadata
recordset_candidates = []
field_candidates = []

for entity_id, entity in all_entities_by_id.items():
    entity_type = entity.get("@type")
    if entity_type == "cr:RecordSet" or entity_type == "RecordSet":
        recordset_candidates.append(entity)
    if entity_type == "cr:Field" or entity_type == "Field":
        field_candidates.append(entity)

print(f"Record set candidates found: {len(recordset_candidates)}")
print(f"Field candidates found: {len(field_candidates)}")

If explicit record sets are present in the metadata, the next cell will display them. If not, we will still attempt to inspect available records directly using `mlcroissant` methods in the next section.

In [ ]:
recordset_overview = []
for rs in recordset_candidates:
    recordset_overview.append({
        "record_set_id": rs.get("@id"),
        "name": rs.get("name"),
        "description": rs.get("description"),
        "field_count": len(rs.get("field", [])) if isinstance(rs.get("field"), list) else 0,
    })

recordset_overview_df = pd.DataFrame(recordset_overview)
recordset_overview_df

In [ ]:
# Preview field metadata if available
field_overview = []
for fld in field_candidates[:50]:
    field_overview.append({
        "field_id": fld.get("@id"),
        "name": fld.get("name"),
        "description": fld.get("description"),
        "dataType": fld.get("dataType"),
        "source": fld.get("source"),
    })

pd.DataFrame(field_overview)

## 3. Record Inspection and Data Extraction
We now inspect records accessible through the dataset.

Since Croissant datasets vary in how record sets are exposed, this notebook uses a robust workflow:

1. Discover record set `@id`s from metadata when possible.
2. Attempt to read records from each discovered record set.
3. Convert the results into pandas DataFrames.
4. Select one or more non-empty DataFrames for exploration.

All record sets are referenced by their `@id` values.

In [ ]:
# Build the list of record set IDs using @id values
record_set_ids = [rs.get("@id") for rs in recordset_candidates if rs.get("@id")]

print("Discovered record set IDs:")
for rs_id in record_set_ids:
    print("-", rs_id)

In [ ]:
# Attempt to read a small sample from each discovered record set
sample_records_by_recordset = {}

for rs_id in record_set_ids:
    try:
        iterator = dataset.records(record_set=rs_id)
        sample = []
        for i, rec in enumerate(iterator):
            sample.append(rec)
            if i >= 2:
                break
        sample_records_by_recordset[rs_id] = sample
        print(f"\nRecord set: {rs_id}")
        print(f"Sample record count retrieved: {len(sample)}")
        if sample:
            print(json.dumps(sample[0], indent=2, default=str)[:1200])
    except Exception as e:
        sample_records_by_recordset[rs_id] = []
        print(f"\nRecord set: {rs_id}")
        print("Could not read sample records:", repr(e))

Next, we load full record sets into pandas DataFrames where possible.

In [ ]:
dataframes = {}
load_report = []

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        load_report.append({
            "record_set_id": rs_id,
            "rows": len(df),
            "columns": len(df.columns),
            "status": "loaded",
        })
    except Exception as e:
        load_report.append({
            "record_set_id": rs_id,
            "rows": None,
            "columns": None,
            "status": f"failed: {repr(e)}",
        })

load_report_df = pd.DataFrame(load_report)
load_report_df

In [ ]:
# Show the loaded record sets with non-empty data
non_empty_record_sets = [rs_id for rs_id, df in dataframes.items() if not df.empty]
print("Non-empty loaded record sets:")
for rs_id in non_empty_record_sets:
    print("-", rs_id, f"({dataframes[rs_id].shape[0]} rows, {dataframes[rs_id].shape[1]} columns)")

In [ ]:
# Select a primary record set dynamically
primary_record_set_id = non_empty_record_sets[0] if non_empty_record_sets else None
primary_df = dataframes[primary_record_set_id].copy() if primary_record_set_id else pd.DataFrame()

print("Primary record set ID:", primary_record_set_id)
print("Shape:", primary_df.shape)
primary_df.head()

The next cell lists the columns of the selected record set. These column names correspond to fields materialized from Croissant field definitions and should be interpreted together with the field metadata when available.

In [ ]:
if primary_record_set_id is not None:
    print("Columns in primary record set:")
    print(primary_df.columns.tolist())
else:
    print("No non-empty record set was loaded.")

## 4. Exploratory Data Analysis (EDA)
This section follows a reusable exploration pattern:

- identify numeric fields dynamically,
- filter rows using a threshold,
- normalize a numeric field,
- group by a categorical field when available,
- inspect missingness and value distributions.

Because this dataset may contain many heterogeneous record sets, the code below automatically chooses candidate fields from the selected record set.

In [ ]:
if primary_record_set_id is not None and not primary_df.empty:
    # Try to coerce object columns to numeric when possible
    analysis_df = primary_df.copy()
    for col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="ignore")

    numeric_columns = analysis_df.select_dtypes(include="number").columns.tolist()
    categorical_columns = analysis_df.select_dtypes(exclude="number").columns.tolist()

    print("Numeric columns:", numeric_columns)
    print("\nCategorical/non-numeric columns:", categorical_columns[:20])
else:
    analysis_df = pd.DataFrame()
    numeric_columns = []
    categorical_columns = []
    print("No data available for EDA.")

In [ ]:
# Select fields dynamically
numeric_field_id = numeric_columns[0] if numeric_columns else None
group_field_id = categorical_columns[0] if categorical_columns else None

print("Selected numeric field:", numeric_field_id)
print("Selected group field:", group_field_id)

In [ ]:
if numeric_field_id is not None:
    numeric_series = pd.to_numeric(analysis_df[numeric_field_id], errors="coerce").dropna()
    if len(numeric_series) > 0:
        threshold = numeric_series.quantile(0.50)
        filtered_df = analysis_df[pd.to_numeric(analysis_df[numeric_field_id], errors="coerce") > threshold].copy()

        print(f"Using threshold based on the median of '{numeric_field_id}': {threshold}")
        print(f"Filtered rows: {len(filtered_df)} out of {len(analysis_df)}")
        display(filtered_df.head())

        std = filtered_df[numeric_field_id].std()
        if pd.notnull(std) and std != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / std
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0.0

        print("\nNormalized values:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        filtered_df = analysis_df.copy()
        print("Selected numeric field contains no usable numeric values.")
else:
    filtered_df = analysis_df.copy()
    print("No numeric field available for filtering and normalization.")

In [ ]:
if numeric_field_id is not None and group_field_id is not None and group_field_id in filtered_df.columns:
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id]
        .agg(["count", "mean", "median", "min", "max"])
        .sort_values("count", ascending=False)
    )
    print(f"Grouped summary by '{group_field_id}':")
    display(grouped_df.head(10))
else:
    grouped_df = pd.DataFrame()
    print("Grouping step skipped because suitable numeric/group fields were not available.")

In [ ]:
# Missingness overview
if not analysis_df.empty:
    missingness_df = (
        analysis_df.isna().sum()
        .rename("missing_count")
        .to_frame()
        .assign(missing_pct=lambda x: x["missing_count"] / len(analysis_df) * 100)
        .sort_values("missing_pct", ascending=False)
    )
    display(missingness_df.head(20))
else:
    print("No data available for missingness analysis.")

## 5. Visualization
We now visualize a selected numeric field and, when possible, compare it across a categorical grouping field.

The plotting code is dynamic so that it works with whatever record set was successfully loaded.

In [ ]:
sns.set_theme(style="whitegrid")

if numeric_field_id is not None and numeric_field_id in analysis_df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(pd.to_numeric(analysis_df[numeric_field_id], errors="coerce").dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for histogram visualization.")

In [ ]:
if (
    numeric_field_id is not None
    and group_field_id is not None
    and group_field_id in analysis_df.columns
    and analysis_df[group_field_id].nunique(dropna=True) > 1
):
    plot_df = analysis_df[[group_field_id, numeric_field_id]].copy()
    plot_df[numeric_field_id] = pd.to_numeric(plot_df[numeric_field_id], errors="coerce")
    plot_df = plot_df.dropna().copy()

    top_groups = plot_df[group_field_id].value_counts().head(10).index
    plot_df = plot_df[plot_df[group_field_id].isin(top_groups)]

    if not plot_df.empty:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top categories)")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough grouped data available for boxplot visualization.")
else:
    print("No suitable categorical grouping field available for boxplot visualization.")

## 6. Optional: Inspect Additional Record Sets
If multiple record sets were loaded successfully, the next cell provides a compact summary of each one so you can choose other tables for further analysis.

In [ ]:
additional_summaries = []
for rs_id, df in dataframes.items():
    additional_summaries.append({
        "record_set_id": rs_id,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names_preview": df.columns.tolist()[:10],
    })

pd.DataFrame(additional_summaries).sort_values(["rows", "columns"], ascending=False)

## 7. Conclusion
In this notebook, we:

- loaded the FAIR²-FoodHealth dataset from its Croissant schema URL,
- inspected dataset-level metadata,
- discovered entities and record sets using their `@id` values,
- loaded available record sets into pandas DataFrames,
- selected a non-empty record set dynamically for analysis,
- performed basic exploratory data analysis, including filtering, normalization, grouping, and missingness inspection,
- visualized numeric distributions and grouped comparisons when suitable fields were available.

### Suggested next steps
- Explore additional record sets by setting `primary_record_set_id` manually to another discovered `@id`.
- Match loaded DataFrame columns back to field metadata discovered from the Croissant schema.
- Build domain-specific analyses for food composition, nutrient relationships, provenance, mappings, or guideline-oriented links depending on the available tables.

Because this dataset is heterogeneous and includes synthetic, curated, and source-derived records, always interpret results in light of the metadata, provenance fields, and documented limitations.